<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/AtividadeProjetoItegrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
!pip -q install spacy nltk scikit-learn pandas matplotlib
!python -m spacy download pt_core_news_md -q

import spacy
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from IPython.display import display

nltk.download("rslp")
nltk.download("punkt")

nlp = spacy.load("pt_core_news_md")

stemmer = RSLPStemmer()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 28.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


##0. Corpus

In [54]:
corpus = [
    "Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.",
    "Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?",
    "Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.",
    "Qual é o valor do condomínio e do IPTU do apartamento no centro de Araraquara?",
    "A casa na Vila Mariana ainda está disponível para locação?",
    "Aceitam fiador ou seguro fiança para o contrato de aluguel?",
    "Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.",
    "O apartamento é novo ou já foi reformado recentemente?",
    "Qual o prazo médio de aprovação do financiamento bancário?",
    "Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à vista.",
    "O sobrado no bairro Santa Cecília aceita animais de estimação?",
    "Poderiam enviar mais fotos do apartamento e da planta baixa?",
    "Qual é a taxa de juros aplicada pelo Banco do Brasil nesse tipo de financiamento?",
    "A casa alugada tem vaga de garagem coberta?",
    "Tenho interesse em investir em um imóvel para locação de longo prazo.",
    "Poderia me indicar imóveis disponíveis na região central de Matão?"
]

docs = [nlp(texto) for texto in corpus]

##1. Tokenização

In [56]:
for i in [0, 1]:
    print(f"\nDocumento {i+1}:")
    print(corpus[i])
    print("\nTokens:")
    print([token.text for token in docs[i]])

tokens_corpus = []
for doc in docs:
    tokens_corpus.append([token.text for token in doc])




Documento 1:
Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.

Tokens:
['Bom', 'dia', '!', 'Meu', 'nome', 'é', 'Marcos', 'Ferreira', 'e', 'tenho', 'interesse', 'no', 'apartamento', 'anunciado', 'na', 'Rua', 'Voluntários', 'da', 'Pátria', ',', 'em', 'Matão', '.']

Documento 2:
Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?

Tokens:
['Vocês', 'trabalham', 'com', 'financiamento', 'pela', 'Caixa', 'Econômica', 'Federal', 'para', 'imóveis', 'usados', '?']


##2. STOPWORDS, STEMMING E LEMATIZAÇÃO

In [57]:
doc_escolhido = docs[0]

linhas = []

for token in doc_escolhido:
    if token.is_punct:
        continue

    if token.is_stop:
        continue

    if not token.text.strip():
        continue

    original = token.text.lower()
    stem = stemmer.stem(original)

    lemma = token.lemma_.lower().strip()
    if not lemma or lemma == "-pron-":
        lemma = original

    linhas.append({
        "Original": original,
        "Stem (RSLP)": stem,
        "Lema (spaCy)": lemma
    })

df_normalizacao = pd.DataFrame(linhas)

display(df_normalizacao)



,Original,Stem (RSLP),Lema (spaCy)
0,dia,dia,dia
1,nome,nom,nome
2,marcos,marc,marcos
3,ferreira,ferr,ferreira
4,interesse,inter,interesse
5,apartamento,apart,apartamento
6,anunciado,anunci,anunciar
7,rua,rua,rua
8,voluntários,voluntári,voluntários
9,pátria,pátr,pátria


##3. ANÁLISE SINTÀTICA

In [61]:
def analisar_sintaxe(doc, numero_documento):
    print(f"\n{'-' * 70}")
    print(f"Documento {numero_documento}")
    print(f"{'-' * 70}")
    print(doc.text)

    # --------------------------------------------------------
    # POS TAGGING
    # --------------------------------------------------------

    dados_pos = []

    for token in doc:
        dados_pos.append({
            "Token": token.text,
            "POS": token.pos_,
            "Tag": token.tag_,
            "Dependência": token.dep_,
            "Head": token.head.text
        })

    df_pos = pd.DataFrame(dados_pos)

    print("\nPOS TAGGING:")
    display(df_pos)

    # --------------------------------------------------------
    # SINTAGMAS NOMINAIS
    # --------------------------------------------------------

    print("Sintagmas Nominais (SN):")

    noun_chunks = list(doc.noun_chunks)

    if noun_chunks:
        for chunk in noun_chunks:
            print(" -", chunk.text)
    else:
        print("Nenhum sintagma nominal identificado.")

    # --------------------------------------------------------
    # SINTAGMAS VERBAIS
    # --------------------------------------------------------

    print("\nSintagmas Verbais (SV):")

    verbos = [token for token in doc if token.pos_ in ("VERB", "AUX")]

    sv_encontrados = []

    for verbo in verbos:
        subtree = list(verbo.subtree)

        if subtree:
            inicio = min(t.i for t in subtree)
            fim = max(t.i for t in subtree)

            texto_sv = doc[inicio:fim + 1].text

            if texto_sv not in sv_encontrados:
                sv_encontrados.append(texto_sv)

    if sv_encontrados:
        for sv in sv_encontrados:
            print(" -", sv)
    else:
        print("Nenhum sintagma verbal identificado.")

    # --------------------------------------------------------
    # ROOT
    # --------------------------------------------------------

    root = next((token for token in doc if token.dep_ == "ROOT"), None)

    print("\nRaiz (ROOT):")

    if root is not None:
        print("Token:", root.text)
        print("POS:", root.pos_)
        print("Dependência:", root.dep_)


        copulas = [
            token for token in doc
            if token.dep_ == "cop"
        ]

        if root.dep_ == "ROOT" and root.pos_ == "VERB":
            print("Comentário: ROOT é um verbo comum da oração.")

        elif root.dep_ == "ROOT" and root.pos_ == "AUX":
            print("Comentário: ROOT está relacionado a uma construção")
            print("verbal com auxiliar.")

        elif copulas:
            print("Comentário: trata-se de uma estrutura com cópula.")
            print("Verbo copulativo identificado:", copulas[0].text)

        else:
            print("Comentário: ROOT não é um verbo comum;")
            print("pode representar uma construção nominal ou elíptica.")

    return df_pos

df_pos_doc1 = analisar_sintaxe(docs[0], 1)
df_pos_doc2 = analisar_sintaxe(docs[6], 7)



----------------------------------------------------------------------
Documento 1
----------------------------------------------------------------------
Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.

POS TAGGING:


,Token,POS,Tag,Dependência,Head
0,Bom,ADJ,ADJ,amod,dia
1,dia,NOUN,NOUN,ROOT,dia
2,!,PUNCT,PUNCT,punct,dia
3,Meu,DET,DET,det,nome
4,nome,NOUN,NOUN,nsubj,Marcos
5,é,AUX,AUX,cop,Marcos
6,Marcos,PROPN,PROPN,ROOT,Marcos
7,Ferreira,PROPN,PROPN,flat:name,Marcos
8,e,CCONJ,CCONJ,cc,tenho
9,tenho,VERB,VERB,conj,Marcos


Sintagmas Nominais (SN):
 - Bom dia
 - Meu nome
 - interesse
 - apartamento
 - Rua Voluntários da Pátria
 - Matão

Sintagmas Verbais (SV):
 - é
 - e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão
 - anunciado na Rua Voluntários da Pátria,

Raiz (ROOT):
Token: dia
POS: NOUN
Dependência: ROOT
Comentário: trata-se de uma estrutura com cópula.
Verbo copulativo identificado: é

----------------------------------------------------------------------
Documento 7
----------------------------------------------------------------------
Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.

POS TAGGING:


,Token,POS,Tag,Dependência,Head
0,Sou,AUX,AUX,cop,corretora
1,a,DET,DET,det,corretora
2,corretora,NOUN,NOUN,ROOT,corretora
3,Camila,PROPN,PROPN,appos,corretora
4,Souza,PROPN,PROPN,flat:name,Camila
5,",",PUNCT,PUNCT,punct,entrando
6,entrando,VERB,VERB,acl,corretora
7,em,ADP,ADP,case,contato
8,contato,NOUN,NOUN,obl,entrando
9,sobre,ADP,ADP,case,interesse


Sintagmas Nominais (SN):
 - Sou a corretora
 - Camila Souza
 - contato
 - o interesse
 - cliente
 - João Pedro
 - imóvel
 - Ribeirão Preto

Sintagmas Verbais (SV):
 - Sou
 - , entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto

Raiz (ROOT):
Token: corretora
POS: NOUN
Dependência: ROOT
Comentário: trata-se de uma estrutura com cópula.
Verbo copulativo identificado: Sou


##4. INTERPRETAçÃO SEMÂNTICA

In [63]:
pares = [
    ("apartamento", "imóvel"),
    ("financiamento", "aluguel")
]

for palavra1, palavra2 in pares:

    doc1 = nlp(palavra1)
    doc2 = nlp(palavra2)

    token1 = doc1[0]
    token2 = doc2[0]

    similaridade = token1.similarity(token2)

    print(f"\n{palavra1} x {palavra2}")
    print(f"Similaridade semântica: {similaridade:.4f}")

    if similaridade >= 0.70:
        print("Comentário: as palavras apresentam alta similaridade")
        print("semântica no modelo utilizado.")
    elif similaridade >= 0.40:
        print("Comentário: as palavras apresentam similaridade")
        print("semântica moderada.")
    else:
        print("Comentário: as palavras apresentam baixa similaridade.")


apartamento x imóvel
Similaridade semântica: 0.6667
Comentário: as palavras apresentam similaridade
semântica moderada.

financiamento x aluguel
Similaridade semântica: 0.3090
Comentário: as palavras apresentam baixa similaridade.


##5. EXTRAçÃO DE FEATURES

In [64]:
corpus_lemmatizado = []

for doc in docs:
    lemas = []

    for token in doc:
        if token.is_punct:
            continue

        if token.is_space:
            continue

        if token.is_stop:
            continue

        if not token.is_alpha:
            continue

        lema = token.lemma_.lower().strip()

        if not lema or lema == "-pron-":
            lema = token.text.lower()

        lemas.append(lema)

    corpus_lemmatizado.append(" ".join(lemas))

print("Corpus lematizado:\n")

for i, texto in enumerate(corpus_lemmatizado, start=1):
    print(f"Documento {i}: {texto}")


# ------------------------------------------------------------
# BAG-OF-WORDS
# ------------------------------------------------------------

vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(corpus_lemmatizado)

termos_bow = vectorizer_bow.get_feature_names_out()

df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=termos_bow,
    index=[f"Doc {i}" for i in range(1, len(corpus) + 1)]
)

print("\nMatriz Bag-of-Words:")
display(df_bow)


# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(corpus_lemmatizado)

termos_tfidf = vectorizer_tfidf.get_feature_names_out()

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=termos_tfidf,
    index=[f"Doc {i}" for i in range(1, len(corpus) + 1)]
)

print("\nMatriz TF-IDF:")
display(df_tfidf.round(4))


# ------------------------------------------------------------
# DOCUMENTO COM MAIS PALAVRAS ÚNICAS
# ------------------------------------------------------------

palavras_unicas = []

for texto in corpus_lemmatizado:
    palavras = texto.split()
    palavras_unicas.append(len(set(palavras)))

df_unicas = pd.DataFrame({
    "Documento": range(1, len(corpus) + 1),
    "Palavras únicas": palavras_unicas
})

print("\nQuantidade de palavras únicas por documento:")
display(df_unicas)

doc_mais_unicas = int(df_unicas.loc[
    df_unicas["Palavras únicas"].idxmax(),
    "Documento"
])

maior_qtd_unicas = int(df_unicas["Palavras únicas"].max())

print(
    f"O documento com mais palavras únicas é o Documento "
    f"{doc_mais_unicas}, com {maior_qtd_unicas} palavras únicas."
)


# ------------------------------------------------------------
# MAIOR PESO TF-IDF
# ------------------------------------------------------------

maior_valor = X_tfidf.max()
linha, coluna = X_tfidf.nonzero()

melhor_doc = None
melhor_termo = None
melhor_peso = 0

for i in range(X_tfidf.shape[0]):
    valores = X_tfidf[i].toarray()[0]

    indice_max = valores.argmax()

    if valores[indice_max] > melhor_peso:
        melhor_peso = valores[indice_max]
        melhor_doc = i + 1
        melhor_termo = termos_tfidf[indice_max]

print(
    f"\nMaior peso TF-IDF encontrado: "
    f"{melhor_peso:.4f}"
)

print(
    f"Documento: {melhor_doc}"
)

print(
    f"Palavra: {melhor_termo}"
)

print(
    "\nComentário: uma palavra recebe peso TF-IDF alto quando possui "
    "relevância dentro do documento e aparece relativamente pouco nos "
    "demais documentos do corpus."
)


Corpus lematizado:

Documento 1: dia nome marcos ferreira interesse apartamento anunciar rua voluntários pátria matão
Documento 2: trabalhar financiamento caixa econômica federal imóvel usar
Documento 3: gostaria agendar visita imóvel bairro jardim europa semana
Documento 4: condomínio iptu apartamento centro araraquara
Documento 5: casa vila mariana disponível locação
Documento 6: aceitam fiador seguro fiança contrato aluguel
Documento 7: corretora camila souza entrar contato interesse cliente joão pedro imóvel ribeirão preto
Documento 8: apartamento reformar recentemente
Documento 9: prazo médio aprovação financiamento bancário
Documento 10: buscar imóvel shopping iguatemi compra vista
Documento 11: sobrado bairro santa cecília aceitar animal estimação
Documento 12: poder enviar foto apartamento planta baixo
Documento 13: taxa juro aplicar banco brasil financiamento
Documento 14: casa alugar vaga garagem cobrir
Documento 15: interesse investir imóvel locação longo prazo
Documento 16:

,aceitam,aceitar,agendar,alugar,aluguel,animal,anunciar,apartamento,aplicar,aprovação,...,sobrado,souza,taxa,trabalhar,usar,vaga,vila,visita,vista,voluntários
Doc 1,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,1
Doc 2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
Doc 3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
Doc 4,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
Doc 5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
Doc 6,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Doc 7,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
Doc 8,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
Doc 9,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
Doc 10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0



Matriz TF-IDF:


,aceitam,aceitar,agendar,alugar,aluguel,animal,anunciar,apartamento,aplicar,aprovação,...,sobrado,souza,taxa,trabalhar,usar,vaga,vila,visita,vista,voluntários
Doc 1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3183,0.2255,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3183
Doc 2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.4093,0.4093,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 3,0.0000,0.0000,0.3748,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3748,0.0000,0.0000
Doc 4,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3338,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 5,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4836,0.0000,0.0000,0.0000
Doc 6,0.4082,0.0000,0.0000,0.0000,0.4082,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 7,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.3019,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 8,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4478,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 9,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4786,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 10,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4319,0.0000



Quantidade de palavras únicas por documento:


,Documento,Palavras únicas
0,1,11
1,2,7
2,3,8
3,4,5
4,5,5
5,6,6
6,7,12
7,8,3
8,9,5
9,10,6


O documento com mais palavras únicas é o Documento 7, com 12 palavras únicas.

Maior peso TF-IDF encontrado: 0.6323
Documento: 8
Palavra: recentemente

Comentário: uma palavra recebe peso TF-IDF alto quando possui relevância dentro do documento e aparece relativamente pouco nos demais documentos do corpus.


##6. DESCOBERTA DE CONHECIMENTO EM TEXTOS (KDT)

In [65]:
# ------------------------------------------------------------
# NER — RECONHECIMENTO DE ENTIDADES NOMEADAS
# ------------------------------------------------------------

print("\nNER — Entidades Nomeadas")

documentos_ner = [0, 6, 9]

for indice in documentos_ner:

    doc = docs[indice]

    print(f"\nDocumento {indice + 1}:")
    print(doc.text)

    if doc.ents:
        for ent in doc.ents:
            print(f" - {ent.text} --> {ent.label_}")
    else:
        print("Nenhuma entidade nomeada identificada.")

print(
    "\nO NER permite identificar automaticamente entidades como "
    "pessoas, locais e organizações presentes nos textos."
)


# ------------------------------------------------------------
# PALAVRAS-CHAVE DE CADA DOCUMENTO
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("PALAVRAS-CHAVE POR DOCUMENTO")
print("-" * 70)

for i in range(len(corpus)):

    valores = X_tfidf[i].toarray()[0]

    indices_ordenados = valores.argsort()[::-1]

    palavras_chave = []

    for indice in indices_ordenados:

        if valores[indice] <= 0:
            continue

        termo = termos_tfidf[indice]

        palavras_chave.append(
            (termo, valores[indice])
        )

        if len(palavras_chave) == 5:
            break

    print(f"\nDocumento {i + 1}:")
    for termo, peso in palavras_chave:
        print(f" - {termo}: {peso:.4f}")


# ------------------------------------------------------------
# LDA
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LDA — DESCOBERTA DE TÓPICOS")
print("-" * 70)

lda = LatentDirichletAllocation(
    n_components=2,
    random_state=42,
    max_iter=20
)

lda.fit(X_bow)


# ------------------------------------------------------------
# PALAVRAS DE CADA TÓPICO
# ------------------------------------------------------------

n_top_words = 10

print("\nPalavras principais de cada tópico:")

for topico, componentes in enumerate(lda.components_, start=1):

    indices = componentes.argsort()[::-1][:n_top_words]

    palavras = [
        termos_bow[i]
        for i in indices
    ]

    print(f"\nTópico {topico}:")
    print(", ".join(palavras))


# ------------------------------------------------------------
# DISTRIBUIÇÃO DOS TÓPICOS POR DOCUMENTO
# ------------------------------------------------------------

distribuicao_topicos = lda.transform(X_bow)

df_lda = pd.DataFrame(
    distribuicao_topicos,
    columns=["Tópico 1", "Tópico 2"],
    index=[f"Doc {i}" for i in range(1, len(corpus) + 1)]
)

print("\nDistribuição dos tópicos por documento:")
display(df_lda.round(4))

print("\nTópico predominante por documento:")

for i, linha in enumerate(distribuicao_topicos, start=1):

    topico_predominante = np.argmax(linha) + 1
    probabilidade = np.max(linha)

    print(
        f"Documento {i}: "
        f"Tópico {topico_predominante} "
        f"({probabilidade:.4f})"
    )



NER — Entidades Nomeadas

Documento 1:
Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.
 - Marcos Ferreira --> PER
 - Rua Voluntários da Pátria --> LOC
 - Matão --> LOC

Documento 7:
Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.
 - Camila Souza --> PER
 - João Pedro --> PER
 - Ribeirão Preto --> LOC

Documento 10:
Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à vista.
 - Shopping Iguatemi --> LOC

O NER permite identificar automaticamente entidades como pessoas, locais e organizações presentes nos textos.

----------------------------------------------------------------------
PALAVRAS-CHAVE POR DOCUMENTO
----------------------------------------------------------------------

Documento 1:
 - voluntários: 0.3183
 - pátria: 0.3183
 - rua: 0.3183
 - dia: 0.3183
 - ferreira: 0.3183

Documento 2:
 - trabalhar: 0.4093
 - usar: 0.4

,Tópico 1,Tópico 2
Doc 1,0.0476,0.9524
Doc 2,0.0696,0.9304
Doc 3,0.0630,0.9370
Doc 4,0.0924,0.9076
Doc 5,0.0915,0.9085
Doc 6,0.9255,0.0745
Doc 7,0.9585,0.0415
Doc 8,0.8532,0.1468
Doc 9,0.0957,0.9043
Doc 10,0.9216,0.0784



Tópico predominante por documento:
Documento 1: Tópico 2 (0.9524)
Documento 2: Tópico 2 (0.9304)
Documento 3: Tópico 2 (0.9370)
Documento 4: Tópico 2 (0.9076)
Documento 5: Tópico 2 (0.9085)
Documento 6: Tópico 1 (0.9255)
Documento 7: Tópico 1 (0.9585)
Documento 8: Tópico 1 (0.8532)
Documento 9: Tópico 2 (0.9043)
Documento 10: Tópico 1 (0.9216)
Documento 11: Tópico 1 (0.9320)
Documento 12: Tópico 1 (0.9214)
Documento 13: Tópico 1 (0.9158)
Documento 14: Tópico 2 (0.9134)
Documento 15: Tópico 1 (0.9046)
Documento 16: Tópico 2 (0.9324)


##7. CONCLUSÃO INTEGRADORA

In [66]:
conclusao = """
As etapas do pipeline de Processamento de Linguagem Natural estão
diretamente relacionadas. A tokenização foi necessária para dividir os
documentos em unidades que pudessem ser analisadas. Em seguida, a
remoção de stopwords, o stemming e a lematização ajudaram a reduzir
variações e palavras pouco relevantes. A análise sintática permitiu
identificar funções gramaticais, sintagmas e raízes das frases,
enquanto a análise semântica possibilitou comparar o significado de
palavras como "apartamento" e "imóvel". Essas etapas prepararam o
corpus para a extração de features, como Bag-of-Words e TF-IDF, que
transformaram os textos em representações numéricas. Por fim, essas
representações puderam ser utilizadas em tarefas de KDT, como
identificação de entidades, extração de palavras-chave e modelagem de
tópicos com LDA. Um exemplo concreto é o uso dos lemas gerados na
etapa de normalização para construir a matriz TF-IDF e, posteriormente,
utilizar a representação do corpus para descobrir tópicos com LDA.
"""

print(conclusao)


As etapas do pipeline de Processamento de Linguagem Natural estão
diretamente relacionadas. A tokenização foi necessária para dividir os
documentos em unidades que pudessem ser analisadas. Em seguida, a
remoção de stopwords, o stemming e a lematização ajudaram a reduzir
variações e palavras pouco relevantes. A análise sintática permitiu
identificar funções gramaticais, sintagmas e raízes das frases,
enquanto a análise semântica possibilitou comparar o significado de
palavras como "apartamento" e "imóvel". Essas etapas prepararam o
corpus para a extração de features, como Bag-of-Words e TF-IDF, que
transformaram os textos em representações numéricas. Por fim, essas
representações puderam ser utilizadas em tarefas de KDT, como
identificação de entidades, extração de palavras-chave e modelagem de
tópicos com LDA. Um exemplo concreto é o uso dos lemas gerados na
etapa de normalização para construir a matriz TF-IDF e, posteriormente,
utilizar a representação do corpus para descobrir tópico